# Novo CAGED geral de 2025

Este notebook consolida os arquivos mensais de movimentações formais. As tabelas de saída servem de referência para comparar o mercado geral com o recorte de migrantes.

In [1]:
from pathlib import Path
import pandas as pd

# Localiza os 12 arquivos mensais do CAGED
pasta = Path("../data/raw/trabalho/caged_geral")
arquivos = sorted(pasta.glob("CAGEDMOV2025*"))

print("Arquivos encontrados:", len(arquivos))
print("Primeiro arquivo:", arquivos[0].name)

# Lê somente cinco registros para conhecer a estrutura
amostra = pd.read_csv(
    arquivos[0],
    sep=";",
    encoding="utf-8",
    nrows=5
)

# Exibe as colunas e os primeiros registros
print("\nColunas:", amostra.columns.tolist())
display(amostra.head())

Arquivos encontrados: 12
Primeiro arquivo: CAGEDMOV202501.txt

Colunas: ['competênciamov', 'região', 'uf', 'município', 'seção', 'subclasse', 'saldomovimentação', 'cbo2002ocupação', 'categoria', 'graudeinstrução', 'idade', 'horascontratuais', 'raçacor', 'sexo', 'tipoempregador', 'tipoestabelecimento', 'tipomovimentação', 'tipodedeficiência', 'indtrabintermitente', 'indtrabparcial', 'salário', 'tamestabjan', 'indicadoraprendiz', 'origemdainformação', 'competênciadec', 'indicadordeforadoprazo', 'unidadesaláriocódigo', 'valorsaláriofixo']


,competênciamov,região,uf,município,seção,subclasse,saldomovimentação,cbo2002ocupação,categoria,graudeinstrução,...,indtrabintermitente,indtrabparcial,salário,tamestabjan,indicadoraprendiz,origemdainformação,competênciadec,indicadordeforadoprazo,unidadesaláriocódigo,valorsaláriofixo
0,202501,1,11,110004,C,1061901,1,413115,101,7,...,0,0,"1800,00",6,0,1,202501,0,5,"1800,00"
1,202501,1,11,110011,G,4784900,-1,519110,101,7,...,0,0,"1600,00",2,0,1,202501,0,5,"1600,00"
2,202501,5,52,520025,N,8211300,-1,848510,101,7,...,0,0,"2000,00",6,0,1,202501,0,5,"2000,00"
3,202501,3,35,355030,N,7820500,-1,422310,101,7,...,0,0,"1710,00",9,0,1,202501,0,5,"1710,00"
4,202501,4,41,410420,N,8211300,-1,782510,101,7,...,0,0,"0,00",4,0,1,202501,0,99,"0,00"


## Agregar admissões e desligamentos

O código lê apenas as colunas necessárias, em blocos de 200 mil linhas. Os valores +1 e −1 identificam admissões e desligamentos. Cada bloco é resumido por município, por município e ocupação CBO e por município e atividade CNAE. O saldo anual é a diferença entre admissões e desligamentos.

In [2]:
# Inclui UF e classificações econômicas na leitura
colunas = [
    "uf",
    "município",
    "saldomovimentação",
    "cbo2002ocupação",
    "seção",
    "subclasse"
]

# Define os agrupamentos de cada tabela
dimensoes = {
    "municipio": ["uf", "município"],
    "cbo": ["uf", "município", "cbo2002ocupação"],
    "cnae": ["uf", "município", "seção", "subclasse"]
}

# Acumula as contagens de admissões e desligamentos por dimensão
acumulados = {nome: [] for nome in dimensoes}

for arquivo in arquivos:
    totais_mes = {"admissoes": 0, "desligamentos": 0}

    # Lê um bloco de cada vez, sem carregar o arquivo inteiro
    for bloco in pd.read_csv(
        arquivo,
        sep=";",
        encoding="utf-8",
        usecols=colunas,
        dtype={
            "município": "string",
            "cbo2002ocupação": "string",
            "subclasse": "string"
        },
        chunksize=200_000
    ):
        # Mantém apenas admissões (+1) e desligamentos (-1)
        bloco = bloco[bloco["saldomovimentação"].isin([1, -1])].copy()

        # Cria indicadores binários de movimentação
        bloco["admissoes"] = bloco["saldomovimentação"].eq(1).astype("int8")
        bloco["desligamentos"] = bloco["saldomovimentação"].eq(-1).astype("int8")

        totais_mes["admissoes"] += int(bloco["admissoes"].sum())
        totais_mes["desligamentos"] += int(bloco["desligamentos"].sum())

        # Agrega as três dimensões separadamente
        for nome, chaves in dimensoes.items():
            resumo = bloco.groupby(chaves, dropna=False)[
                ["admissoes", "desligamentos"]
            ].sum()

            acumulados[nome].append(resumo)

    print(
        f"{arquivo.name}: "
        f"{totais_mes['admissoes']:,} admissões | "
        f"{totais_mes['desligamentos']:,} desligamentos"
    )

# Consolida os blocos e os meses em tabelas anuais
resultados_caged_geral = {}

for nome, partes in acumulados.items():
    tabela = pd.concat(partes).groupby(
        level=list(range(len(dimensoes[nome]))),
        dropna=False
    ).sum().reset_index()

    # Calcula o saldo anual
    tabela["saldo"] = tabela["admissoes"] - tabela["desligamentos"]

    resultados_caged_geral[nome] = tabela

    print(f"\n{nome}: {len(tabela):,} combinações")
    print("Admissões:", tabela["admissoes"].sum())
    print("Desligamentos:", tabela["desligamentos"].sum())

CAGEDMOV202501.txt: 2,271,611 admissões | 2,134,308 desligamentos
CAGEDMOV202502.txt: 2,579,192 admissões | 2,147,197 desligamentos
CAGEDMOV202503.txt: 2,234,662 admissões | 2,163,086 desligamentos
CAGEDMOV202504.txt: 2,282,187 admissões | 2,024,659 desligamentos
CAGEDMOV202505.txt: 2,256,225 admissões | 2,107,233 desligamentos
CAGEDMOV202506.txt: 2,139,182 admissões | 1,972,561 desligamentos
CAGEDMOV202507.txt: 2,251,440 admissões | 2,121,665 desligamentos
CAGEDMOV202508.txt: 2,239,895 admissões | 2,092,537 desligamentos
CAGEDMOV202509.txt: 2,292,492 admissões | 2,079,490 desligamentos
CAGEDMOV202510.txt: 2,271,460 admissões | 2,186,313 desligamentos
CAGEDMOV202511.txt: 1,979,902 admissões | 1,894,038 desligamentos
CAGEDMOV202512.txt: 1,523,309 admissões | 2,141,473 desligamentos

municipio: 5,567 combinações
Admissões: 26321557
Desligamentos: 25064560

cbo: 994,847 combinações
Admissões: 26321557
Desligamentos: 25064560

cnae: 588,597 combinações
Admissões: 26321557
Desligamentos: 25

## Associar os códigos municipais do IBGE

A correspondência usa os seis primeiros dígitos do código IBGE para encontrar a chave do CAGED. O código verifica se a referência é única e mede quantas movimentações ficaram sem correspondência.

In [3]:
# Carrega a referência municipal do IBGE
municipios_ibge = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Cria a chave de seis dígitos utilizada pelo CAGED
municipios_ibge["codigo_caged"] = (
    municipios_ibge["codigo_ibge"]
    .astype(str)
    .str[:6]
)

# Verifica se a chave de seis dígitos identifica municípios unicamente
print("Códigos CAGED duplicados na referência:",
      municipios_ibge["codigo_caged"].duplicated().sum())

# Prepara a tabela municipal do CAGED geral
emprego_geral_municipal_2025 = resultados_caged_geral["municipio"].copy()

# Padroniza o código de seis dígitos
emprego_geral_municipal_2025["município"] = (
    emprego_geral_municipal_2025["município"].str.zfill(6)
)

# Associa o código IBGE completo
emprego_geral_municipal_2025 = emprego_geral_municipal_2025.merge(
    municipios_ibge[["codigo_caged", "codigo_ibge"]],
    left_on="município",
    right_on="codigo_caged",
    how="left",
    validate="many_to_one"
)

# Verifica a correspondência territorial
print("Municípios sem código IBGE:",
      emprego_geral_municipal_2025["codigo_ibge"].isna().sum())

print("Admissões sem código IBGE:",
      emprego_geral_municipal_2025.loc[
          emprego_geral_municipal_2025["codigo_ibge"].isna(),
          "admissoes"
      ].sum())

# Exibe eventuais códigos não identificados
display(
    emprego_geral_municipal_2025[
        emprego_geral_municipal_2025["codigo_ibge"].isna()
    ]
)

Códigos CAGED duplicados na referência: 0
Municípios sem código IBGE: 1
Admissões sem código IBGE: 3272


,uf,município,admissoes,desligamentos,saldo,codigo_caged,codigo_ibge
5566,99,999999,3272,2978,294,NaN,NaN


## Preparar as tabelas territoriais

O código remove o município 999999 e associa o código IBGE completo às três tabelas. Os totais de admissões e desligamentos devem coincidir entre as três agregações.

In [4]:
# Prepara a tabela de correspondência CAGED → IBGE
referencia_municipios = municipios_ibge[
    ["codigo_caged", "codigo_ibge"]
].copy()

# Cria tabelas padronizadas para as três dimensões
tabelas_caged_geral = {}

for nome, tabela in resultados_caged_geral.items():

    # Remove registros sem município identificado
    df = tabela[tabela["município"] != "999999"].copy()

    # Associa o código IBGE de sete dígitos
    df = df.merge(
        referencia_municipios,
        left_on="município",
        right_on="codigo_caged",
        how="left",
        validate="many_to_one"
    )

    # Remove códigos auxiliares e mantém o código IBGE
    df = df.drop(columns=["município", "codigo_caged"])

    tabelas_caged_geral[nome] = df

    # Valida a correspondência e os totais
    print(f"\n{nome}")
    print("Linhas:", len(df))
    print("Sem código IBGE:", df["codigo_ibge"].isna().sum())
    print("Admissões:", df["admissoes"].sum())
    print("Desligamentos:", df["desligamentos"].sum())


municipio
Linhas: 5566
Sem código IBGE: 0
Admissões: 26318285
Desligamentos: 25061582

cbo
Linhas: 994623
Sem código IBGE: 0
Admissões: 26318285
Desligamentos: 25061582

cnae
Linhas: 588508
Sem código IBGE: 0
Admissões: 26318285
Desligamentos: 25061582


## Exportar os resultados

São gravados três arquivos Parquet com os indicadores anuais por município, por município e CBO e por município e CNAE.

In [5]:
from pathlib import Path

# Define a pasta dos dados processados
pasta_saida = Path("../data/processed/trabalho")
pasta_saida.mkdir(parents=True, exist_ok=True)

# Define os nomes dos três arquivos
nomes_arquivos = {
    "municipio": "emprego_geral_municipal_2025.parquet",
    "cbo": "emprego_geral_municipio_cbo_2025.parquet",
    "cnae": "emprego_geral_municipio_cnae_2025.parquet"
}

# Salva as tabelas em formato Parquet
for nome, arquivo in nomes_arquivos.items():
    caminho = pasta_saida / arquivo

    tabelas_caged_geral[nome].to_parquet(
        caminho,
        index=False
    )

    print(f"Salvo: {caminho}")

Salvo: ..\data\processed\trabalho\emprego_geral_municipal_2025.parquet
Salvo: ..\data\processed\trabalho\emprego_geral_municipio_cbo_2025.parquet
Salvo: ..\data\processed\trabalho\emprego_geral_municipio_cnae_2025.parquet
